# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a structure for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the FAIR² Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {getattr(metadata, 'name', '')}\n")
print(f"Description: {getattr(metadata, 'description', '')}\n")
print(f"Published: {getattr(metadata, 'datePublished', '')}\n")
print(f"License: {getattr(metadata, 'license', '')}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. We first list all record set `@id`s found in the dataset's metadata. Fields and columns belonging to each record set are also listed by their `@id` for precise reference when extracting data.

In [ ]:
# List and display all record sets, fields, and columns by their @id

record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    # For older croissant schemas, try 'record_sets'
    record_sets = getattr(metadata, 'record_sets', [])

def get_entity_id(entity):
    if isinstance(entity, dict):
        return entity.get('@id')
    try:
        return getattr(entity, '@id', str(entity))
    except Exception:
        return str(entity)

valid_record_set_ids = []
for rs in record_sets:
    rs_id = get_entity_id(rs)
    valid_record_set_ids.append(rs_id)
    print(f"RecordSet @id: {rs_id}")
    fields = getattr(rs, 'field', getattr(rs, 'fields', []))
    if fields:
        print("Fields and their @id:")
        for f in fields:
            print(f"  - {get_entity_id(f)}")
    columns = getattr(rs, 'column', getattr(rs, 'columns', []))
    if columns:
        print("Columns and their @id:")
        for c in columns:
            print(f"  - {get_entity_id(c)}")
    print()

*If the above list is empty, refer to the Croissant schema JSON for the available record set `@id`(s).*
Below, display sample records for each record set using their `@id`.

In [ ]:
# Display sample records from each record set by @id
for rs_id in valid_record_set_ids:
    print(f"\nSample records for RecordSet @id={rs_id}:")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            print(rec)
            if i >= 2:
                break
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Prepare DataFrames for each record set
# Use the record set @id(s) found above

# Example placeholder RecordSet @id (substitute the actual one from above cell if available):
record_sets_to_use = valid_record_set_ids # use detected RecordSet @ids
dataframes = {}

for rs_id in record_sets_to_use:
    print(f"Loading DataFrame for RecordSet @id={rs_id} ...")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records.")
            print(f"Fields: {dataframes[rs_id].columns.tolist()}")
            display(dataframes[rs_id].head())
        else:
            print("No records found.")
    except Exception as e:
        print(f"  Could not extract DataFrame for {rs_id}: {e}")

# For subsequent analysis, if there is at least one data frame, select the first RecordSet @id
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # Use this for variable placeholders below
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# This cell assumes you have already loaded a DataFrame for analysis (`main_record_set_id` and `dataframes`)

import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Identify candidate numeric fields (columns with dtype float or int)
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields detected for EDA: {numeric_fields}")
    
    # Pick the first numeric field for demonstration
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Analyzing numeric field: {numeric_field}")

        # Set a sample threshold
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by the first categorical column available (if exists)
        group_fields = df.select_dtypes(include=['object']).columns.tolist()
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouped data by {group_field} (mean):")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No categorical field available to group by.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No main DataFrame extracted.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example: Visualize the distribution of the primary numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If both group_field and numeric_field exist, make a boxplot
    if group_fields:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field} (Boxplot)")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
We loaded, inspected, and performed preliminary analysis and visualization of the FAIR² mass spectrometry dataset with `mlcroissant`, using Croissant entity `@id` references for robust and reproducible workflows.

- Used only `@id` references for all RecordSets, fields, and columns
- Loaded metadata and tabular data
- Performed data filtering, normalization, and group-wise analysis
- Visualized primary data distributions

This notebook provides a reproducible blueprint for deeper interpretation and further machine learning or bioinformatics tasks using the FAIR² dataset.
